In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import butter, filtfilt
from pyedflib import EdfReader
from pathlib import Path
import os
import cv2  # ### NOVO ### - Para manipulação e salvamento de imagens
import shutil # ### NOVO ### - Para operações de arquivo (apagar pastas)
from sklearn.model_selection import train_test_split # ### NOVO ### - Para dividir os sujeitos

# --- Funções de Processamento de Dados (Mantidas) ---
# Suas funções bandpass_filter, notch_filter e compute_individual_epoch_stft
# permanecem exatamente as mesmas. Vou omiti-las aqui por brevidade.
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def notch_filter(data, notch_freq, fs, quality=30):
    nyquist = 0.5 * fs
    freq = notch_freq / nyquist
    b, a = butter(2, [freq - 0.5/quality, freq + 0.5/quality], btype='bandstop')
    return filtfilt(b, a, data)

def compute_individual_epoch_stft(data, events, event_type, tmin, tmax, fs,
                                  nperseg, noverlap, nfft):
    event_samples = [e[0] for e in events if e[2] == event_type]
    individual_stft_data = []
    for i, sample in enumerate(event_samples):
        start = sample + int(tmin * fs)
        end = sample + int(tmax * fs)
        epoch = data[start:end]
        if len(epoch) < nperseg:
            continue
        f, t, Zxx = signal.stft(epoch, fs=fs, window='hann',
                                nperseg=nperseg, noverlap=noverlap, nfft=nfft)
        individual_stft_data.append({'epoch_idx': i, 'frequencies': f, 'times': t, 'magnitude': np.abs(Zxx)})
    return individual_stft_data


# ### NOVO ### - Função para salvar espectrogramas como imagem
def save_spectrograms_as_image(c3_magnitude, c4_magnitude, output_path):
    """
    Normaliza os espectrogramas de C3 e C4, os combina em uma imagem de 3 canais
    e a salva como um arquivo .png.
    - C3 -> Canal Vermelho
    - C4 -> Canal Verde
    - Canal Azul -> Fixo em zero
    """
    # Aplicar escala logarítmica para comprimir a faixa dinâmica. Adiciona-se 1 para evitar log(0).
    log_c3 = np.log1p(c3_magnitude)
    log_c4 = np.log1p(c4_magnitude)

    # Normalizar cada canal para o intervalo 0-255
    # cv2.normalize(source, destination, alpha, beta, norm_type)
    # alpha = valor mínimo (0), beta = valor máximo (255)
    norm_c3 = cv2.normalize(log_c3, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    norm_c4 = cv2.normalize(log_c4, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    
    # Criar um canal azul vazio (matriz de zeros) com as mesmas dimensões
    blue_channel = np.zeros_like(norm_c3)

    # Empilhar os canais para formar uma imagem BGR (padrão do OpenCV)
    # Ordem: Blue, Green, Red
    bgr_image = cv2.merge([blue_channel, norm_c4, norm_c3])
    
    # Salvar a imagem
    cv2.imwrite(output_path, bgr_image)

# --- Configurações Globais ---

pasta_raiz_sujeitos = Path('data/phisionet_data') 
start_subject = 1
end_subject = 10
registros = ['R04', 'R08', 'R12']
event_dict = {'T0': 'rest', 'T1': 'left', 'T2': 'right'}
epoch_duration_sec = 4
fs = 160
window_durations_ms = [200]

# ### NOVO ### - Definição dos diretórios de saída e divisão de dados
# O nome do dataset principal será baseado na janela
base_output_data_dir = f"eeg_spectrograms_yolo_{window_durations_ms[0]}ms"

# Limpa o diretório de saída se ele já existir para garantir um dataset limpo
if os.path.exists(base_output_data_dir):
    print(f"Limpando diretório existente: {base_output_data_dir}")
    shutil.rmtree(base_output_data_dir)

# Lista de todos os sujeitos a serem processados
all_subject_ids = [f'S{i:03d}' for i in range(start_subject, end_subject + 1)]

# Dividir os SUJEITOS em conjuntos de treino e validação (80% treino, 20% validação)
# Usar um random_state para garantir que a divisão seja sempre a mesma
train_subjects, val_subjects = train_test_split(all_subject_ids, test_size=0.2, random_state=42)

print(f"Total de sujeitos: {len(all_subject_ids)}")
print(f"Sujeitos de Treino ({len(train_subjects)}): {train_subjects[:5]}...")
print(f"Sujeitos de Validação ({len(val_subjects)}): {val_subjects[:5]}...")

# Criar a estrutura de pastas necessária para YOLO
for split in ['train', 'val']:
    for class_name in event_dict.values():
        os.makedirs(os.path.join(base_output_data_dir, split, class_name), exist_ok=True)


print("\nIniciando a geração de spectrograms para o formato YOLO...")

# --- Loop Principal para Processamento (Aninhado) ---

for current_window_ms in window_durations_ms:
    current_window_s = current_window_ms / 1000.0
    nperseg = int(fs * current_window_s)
    noverlap = int(nperseg * 0.75) 
    nfft = nperseg * 2
    nperseg = max(1, nperseg)
    noverlap = min(noverlap, nperseg - 1)
    if nperseg == 1:
        noverlap = 0

    print(f"\n--- Processando com Janela de {current_window_ms}ms (nperseg={nperseg}, noverlap={noverlap}) ---")

    # ### MODIFICADO ### - O loop agora itera sobre a lista completa de sujeitos
    for subject_id in all_subject_ids:
        subject_folder_path = pasta_raiz_sujeitos / subject_id

        if not subject_folder_path.is_dir():
            print(f" Pasta do sujeito {subject_id} não encontrada. Pulando.")
            continue

        print(f"  Processando Sujeito: {subject_id}")

        # ### NOVO ### - Determinar se o sujeito é de treino ou validação
        if subject_id in train_subjects:
            split_folder = 'train'
        else:
            split_folder = 'val'

        for registro in registros:
            edf_file = next(subject_folder_path.glob(f'*{registro}.edf'), None)
            if not edf_file:
                continue

            reader = EdfReader(str(edf_file))
            n_channels = reader.signals_in_file
            signal_labels = reader.getSignalLabels()
            
            try:
                c3_idx = signal_labels.index('C3..')
                c4_idx = signal_labels.index('C4..')
            except ValueError:
                reader.close()
                continue

            signals_data = np.array([reader.readSignal(c) for c in range(n_channels)])
            c3_signal = signals_data[c3_idx, :]
            c4_signal = signals_data[c4_idx, :]
            annotations_onset, _, annotations_description = reader.readAnnotations()
            reader.close()
            
            events = [[int(onset * fs), 0, desc] for onset, desc in zip(annotations_onset, annotations_description)]
            c3_filtered = notch_filter(bandpass_filter(c3_signal, 0.5, 40, fs), 50, fs)
            c4_filtered = notch_filter(bandpass_filter(c4_signal, 0.5, 40, fs), 50, fs)
            
            conditions = ['T0', 'T1', 'T2']
            for cond_key in conditions:
                class_name = event_dict[cond_key]
                
                # Calcular STFTs para C3 e C4
                c3_epoch_data = compute_individual_epoch_stft(c3_filtered, events, cond_key, tmin=0, tmax=epoch_duration_sec, fs=fs,
                                                               nperseg=nperseg, noverlap=noverlap, nfft=nfft)
                c4_epoch_data = compute_individual_epoch_stft(c4_filtered, events, cond_key, tmin=0, tmax=epoch_duration_sec, fs=fs,
                                                               nperseg=nperseg, noverlap=noverlap, nfft=nfft)

                # ### MODIFICADO ### - Loop para salvar como imagem combinada
                # Assumimos que o número de épocas para C3 e C4 é o mesmo.
                for j in range(len(c3_epoch_data)):
                    # Obter as matrizes de magnitude para a época j
                    Zxx_c3 = c3_epoch_data[j]['magnitude']
                    Zxx_c4 = c4_epoch_data[j]['magnitude']
                    
                    # Garantir que ambas as épocas são válidas
                    if Zxx_c3 is not None and Zxx_c4 is not None:
                        epoch_idx = c3_epoch_data[j]['epoch_idx']
                        
                        # Definir nome do arquivo de saída
                        base_filename = f"{subject_id}_{edf_file.stem.replace('.', '_')}_epoch_{epoch_idx+1}.png"
                        
                        # Montar o caminho de saída completo
                        # Ex: eeg_spectrograms_yolo_200ms/train/left/S001_R04_epoch_1.png
                        output_path = os.path.join(base_output_data_dir, split_folder, class_name, base_filename)
                        
                        # Chamar a nova função para salvar a imagem
                        save_spectrograms_as_image(Zxx_c3, Zxx_c4, output_path)

print(f"\nProcessamento concluído. Dataset para YOLO salvo em: '{base_output_data_dir}'")

Total de sujeitos: 10
Sujeitos de Treino (8): ['S006', 'S001', 'S008', 'S003', 'S010']...
Sujeitos de Validação (2): ['S009', 'S002']...

Iniciando a geração de spectrograms para o formato YOLO...

--- Processando com Janela de 200ms (nperseg=32, noverlap=24) ---
  Processando Sujeito: S001
  Processando Sujeito: S002
  Processando Sujeito: S003
  Processando Sujeito: S004
  Processando Sujeito: S005
  Processando Sujeito: S006
  Processando Sujeito: S007
  Processando Sujeito: S008
  Processando Sujeito: S009
  Processando Sujeito: S010

Processamento concluído. Dataset para YOLO salvo em: 'eeg_spectrograms_yolo_200ms'
